# 01 — Data Split

Walks `planet_superdove_landmasked/` to build a master file inventory,
then performs a **leave-one-site-out** split and writes
`data/splits/train.csv` and `data/splits/val.csv`.

These CSVs are the sole input to `BleachDataset` — run this notebook
once before training or any analysis notebook.

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
DATA_ROOT   = "/path/to/planet_superdove_landmasked"  # root of masked tiles
OUT_DIR     = "data/splits"                           # where CSVs are written

# Site to hold out for validation (leave-one-site-out).
# Change to any site name in SITES below to generate a different fold.
VAL_SITE    = "cheeca_flkeys"

# Sites included in the dataset. lbcaye_bbr excluded (too few tiles).
SITES = [
    "cheeca_flkeys",
    "looe_flkey",
    "easterndry_flkeys",
    "sand_flkey",
    "rock_flkey",
    "sanagustin_mexico",
    "chachacual_mexico",
    "jicaral_mexico",
    "riscalillo_mexico",
    "santacruz_mexico",
    "northpoint_lizard",
    "pulau_kapas",
]

In [ ]:
import os
import glob
import pandas as pd

# ── Build master file inventory ───────────────────────────────────────────────
rows = []
for site in SITES:
    pattern = os.path.join(DATA_ROOT, site, "**", "*.tif")
    for fp in glob.glob(pattern, recursive=True):
        parts = os.path.normpath(fp).split(os.sep)
        # expected layout: .../site/label/tiled_360m/date/locXXX.tif
        try:
            site_idx  = parts.index(site)
            label     = parts[site_idx + 1]   # healthy | bleached
            date      = parts[site_idx + 3]   # YYYYMMDD
        except (ValueError, IndexError):
            continue
        rows.append({
            "site":     site,
            "label":    label,
            "date":     date,
            "filename": os.path.basename(fp),
            "filepath": fp,
        })

assert rows, f"No .tif files found under {DATA_ROOT}. Check DATA_ROOT."

df = pd.DataFrame(rows)
df["image_id"] = range(len(df))
df = df[["site", "label", "date", "filename", "filepath", "image_id"]]

print(f"Total tiles: {len(df)}")
print(df.groupby(["site", "label"]).size().unstack(fill_value=0))

In [ ]:
# ── Leave-one-site-out split ──────────────────────────────────────────────────
assert VAL_SITE in df["site"].unique(), \
    f"VAL_SITE '{VAL_SITE}' not found. Available: {sorted(df['site'].unique())}"

df_val   = df[df["site"] == VAL_SITE].copy()
df_train = df[df["site"] != VAL_SITE].copy()

# Sanity: no tile should appear in both splits
assert set(df_val["image_id"]).isdisjoint(set(df_train["image_id"])), \
    "Split leakage detected!"

print(f"Val site : {VAL_SITE}")
print(f"  val   : {len(df_val):>5} tiles  "
      f"(healthy={( df_val['label']=='healthy').sum()}, "
      f"bleached={(df_val['label']=='bleached').sum()})")
print(f"  train : {len(df_train):>5} tiles  "
      f"(healthy={(df_train['label']=='healthy').sum()}, "
      f"bleached={(df_train['label']=='bleached').sum()})")

In [ ]:
# ── Write CSVs ────────────────────────────────────────────────────────────────
os.makedirs(OUT_DIR, exist_ok=True)

sort_cols = ["site", "label", "date", "filename"]
df_train.sort_values(sort_cols).to_csv(os.path.join(OUT_DIR, "train.csv"), index=False)
df_val.sort_values(sort_cols).to_csv(os.path.join(OUT_DIR, "val.csv"),   index=False)

print(f"Saved train.csv ({len(df_train)} rows) and val.csv ({len(df_val)} rows) to {OUT_DIR}/")